# Borrador histórico

Este cuaderno conserva una exploración previa. Para la entrega y para la presentación usamos [Proyecto_Final_B_Portafolio_Cine.ipynb](../Proyecto_Final_B_Portafolio_Cine.ipynb). Los montos, filtros y propuestas de este borrador pueden ser diferentes y no se deben combinar con los de la versión final.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

raiz = Path.cwd().resolve()
if not (raiz / 'datos' / 'peliculas.csv').exists():
    raiz = raiz.parent

tipos_peliculas = {
    'id_pelicula':'int64', 'titulo':'string', 'anio_estreno':'int64',
    'fecha_estreno':'string', 'genero':'string', 'clasificacion':'string',
    'pais_origen':'string', 'es_secuela':'string', 'director':'string',
    'guionista':'string', 'protagonista':'string', 'productora':'string',
    'duracion_minutos':'float64', 'puntaje_imdb':'float64',
    'cantidad_votos':'float64', 'presupuesto_usd':'float64',
    'recaudacion_mundial_usd':'float64'
}
peliculas = pd.read_csv(raiz / 'datos' / 'peliculas.csv', dtype=tipos_peliculas)
ipc = pd.read_csv(raiz / 'datos' / 'ipc_eeuu.csv',
                  dtype={'anio':'int64','ipc_eeuu':'float64'})

peliculas['fecha_convertida'] = pd.to_datetime(
    peliculas.fecha_estreno, format='mixed', dayfirst=True, errors='coerce')

print('Películas:',len(peliculas),'filas y',len(tipos_peliculas),'columnas originales')
print('IPC:',ipc.shape[0],'filas y',ipc.shape[1],'columnas')
print('Fechas que no se pudieron convertir:',peliculas.fecha_convertida.isna().sum())

display(peliculas.head(3))
display(ipc.head(3))

# Proyecto final B — Etapas 1 a 5

**Programación Analítica II · Arroyo Capital**

Este cuaderno trabaja la auditoría de datos de la etapa 1, la creación de variables de la etapa 2, el análisis de negocio de la etapa 3, las visualizaciones de la etapa 4 y una tesis de inversión condicionada en la etapa 5. Los archivos originales de `datos/` se leen sin modificarlos; las correcciones se aplican a una copia en memoria.

Ejecutar de arriba abajo desde la raíz del proyecto o desde `notebooks/`. Las tablas se muestran con `display`, como en las clases de pandas.

## 1.1 Cargar los dos archivos, declarar tipos y fechas

Declaramos los tipos al leer los CSV. Las fechas mezclan `AAAA-MM-DD` y `DD/MM/AAAA`, por lo que se convierten explícitamente con `format="mixed"` y `dayfirst=True`, tal como se trabajó en EDA. Conservamos también el texto original para auditarlo.

In [ ]:
esquema = pd.concat([
    peliculas.dtypes.rename('tipo').to_frame().assign(archivo='peliculas'),
    ipc.dtypes.rename('tipo').to_frame().assign(archivo='ipc')
])

display(esquema)

categoricas = ['genero','clasificacion','pais_origen','es_secuela',
               'director','guionista','protagonista','productora']

display(peliculas[categoricas].nunique(dropna=False).rename('categorias').to_frame())

continuas = ['duracion_minutos','puntaje_imdb','cantidad_votos',
             'presupuesto_usd','recaudacion_mundial_usd']

display(peliculas[continuas].describe().T.style.format("{:.2f}"))

**Comprobación:** ambos archivos se cargaron y las primeras filas permiten verificar nombres y contenido de las columnas. `fecha_convertida` es una columna auxiliar; no altera `fecha_estreno`.

## 1.2 Dimensiones, tipos, categorías y descripción numérica

Separamos las variables categóricas de las continuas. Los identificadores y el año no se interpretan como medidas continuas.

In [ ]:
ID_ADICIONAL_INICIO = 90001
ID_ADICIONAL_FIN = 90014

# La fecha auxiliar no interviene al comparar las filas originales.
duplicada_exacta = peliculas.drop(columns='fecha_convertida').duplicated()
sin_exactas = peliculas.loc[~duplicada_exacta].copy()
ids_conflictivos = sin_exactas.duplicated(subset='id_pelicula').sum()

# Una película puede aparecer dos veces con distintos ID.
repetida_pelicula = sin_exactas.duplicated(
    subset=['titulo', 'anio_estreno'], keep=False)
peliculas_repetidas = sin_exactas.loc[repetida_pelicula].sort_values(
    ['titulo', 'id_pelicula'])
pares_repetidos = peliculas_repetidas[
    ['titulo', 'anio_estreno']].drop_duplicates()

coinciden_otros_datos = peliculas_repetidas.drop(
    columns=['id_pelicula', 'presupuesto_usd']).duplicated(keep=False)

print('Filas exactamente duplicadas:', duplicada_exacta.sum(),
      f'({duplicada_exacta.mean():.2%} de la base original)')
print('ID repetidos con datos distintos:', ids_conflictivos)
print('Películas repetidas con distinto ID y presupuesto:', len(pares_repetidos),
      'pares /', len(peliculas_repetidas), 'filas')
print('Filas que coinciden salvo ID y presupuesto:',
      coinciden_otros_datos.sum(), 'de', len(peliculas_repetidas))

display(peliculas_repetidas[
    ['id_pelicula', 'titulo', 'anio_estreno', 'presupuesto_usd']])

# Decisión: quitamos el bloque 90001-90014 para contar cada película una vez.
version_adicional = (repetida_pelicula
                     & sin_exactas.id_pelicula.between(
                         ID_ADICIONAL_INICIO, ID_ADICIONAL_FIN))
datos = sin_exactas.loc[~version_adicional].copy()


print('Versiones adicionales retiradas:', version_adicional.sum(),
      f'({version_adicional.sum() / len(peliculas):.2%} de la base original)')
print('Filas antes:', len(peliculas),
      '| después de copias exactas:', len(sin_exactas),
      '| después de películas repetidas:', len(datos))
print('Porcentaje total retirado:',
      f'{(len(peliculas) - len(datos)) / len(peliculas):.2%}')
print('Base conservada: antes 100.00% | tras copias exactas',
      f'{len(sin_exactas) / len(peliculas):.2%}',
      '| tras películas repetidas', f'{len(datos) / len(peliculas):.2%}')
print('Películas repetidas que quedan:',
      datos.duplicated(subset=['titulo', 'anio_estreno']).sum())

Observaciones: el mínimo y el máximo de `describe` ayudan a detectar valores imposibles, que revisamos en el punto 1.5. El número de categorías todavía incluye variantes de escritura.

## 1.3 Filas duplicadas y películas repetidas

Primero buscamos copias idénticas. Luego buscamos películas con el mismo título y año, aunque tengan identificadores distintos. Son problemas diferentes: revisar solo `id_pelicula` no detecta todas las películas repetidas.

Decisión: quitamos las 58 copias exactas porque no aportan información (0,75 % de 7740 filas). También retiramos 14 versiones adicionales (0,18 % de la base original): tienen los ID consecutivos 90001–90014, fuera del bloque principal, y cada una coincide con otra película en todos los campos salvo ID y presupuesto. Conservamos una versión por película para no contarla dos veces. Sin una fuente externa no se puede asegurar cuál de los dos presupuestos es el verdadero; el criterio elegido queda explícito para poder revisarlo.

In [ ]:
faltantes_originales = pd.DataFrame({
    'n_faltantes':datos.isna().sum(),
    'porcentaje':(100*datos.isna().mean()).round(2)
}).sort_values('n_faltantes',ascending=False)

display(faltantes_originales)

# Mostramos ejemplos de las filas filtradas por las que no tienen presupuesto.
display(datos.loc[datos.presupuesto_usd.isna(),
                  ['id_pelicula','titulo','anio_estreno','genero','presupuesto_usd']]
        .head(6))

## 1.4 Faltantes por columna y ejemplos filtrados

Esta tabla cuenta faltantes originales después de quitar las copias exactas y las versiones adicionales de películas repetidas. Los ceros y valores negativos sospechosos todavía figuran como datos; se auditan en el siguiente punto.

In [ ]:
DURACION_NO_INFORMADA = 999
DURACIONES_INVALIDAS = [0, DURACION_NO_INFORMADA]
PUNTAJE_IMDB_MIN = 1
PUNTAJE_IMDB_MAX = 10
ANIO_INICIAL_DATOS = 1980
ANIO_FINAL_DATOS = 2020

datos_sin_corregir = datos.copy()
reglas = {
    'presupuesto_usd':datos.presupuesto_usd <= 0,
    'recaudacion_mundial_usd':datos.recaudacion_mundial_usd < 0,
    'duracion_minutos':datos.duracion_minutos.isin(DURACIONES_INVALIDAS),
    'puntaje_imdb':(~datos.puntaje_imdb.between(
        PUNTAJE_IMDB_MIN, PUNTAJE_IMDB_MAX)
                     & datos.puntaje_imdb.notna()),
    'cantidad_votos':datos.cantidad_votos <= 0
}

filas_cambios = []
for columna, mascara in reglas.items():
    # Guardamos el estado anterior para cuantificar exactamente la corrección.
    antes_pct = 100*datos[columna].isna().mean()
    afectados = int(mascara.sum())
    # Invalidamos solo la celda errónea y conservamos la película.
    datos.loc[mascara,columna] = np.nan
    filas_cambios.append({
        'columna':columna,'valores_invalidados':afectados,
        'base_afectada_pct':round(100*afectados/len(datos),2),
        'faltantes_antes_pct':round(antes_pct,2),
        'faltantes_despues_pct':round(100*datos[columna].isna().mean(),2)
    })

display(pd.DataFrame(filas_cambios))
print('Películas antes y después de invalidar valores:',
      len(datos_sin_corregir), 'y', len(datos))

# Desglosamos cada regla con los valores anteriores a la corrección.
print('Presupuestos en cero:', (datos_sin_corregir.presupuesto_usd == 0).sum(),
      '| negativos:', (datos_sin_corregir.presupuesto_usd < 0).sum())
print('Recaudaciones negativas:',
      (datos_sin_corregir.recaudacion_mundial_usd < 0).sum())
print('Duraciones de 0 minutos:',
      (datos_sin_corregir.duracion_minutos == 0).sum(),
      '| de 999 minutos:',
      (datos_sin_corregir.duracion_minutos == DURACION_NO_INFORMADA).sum())
print('Puntajes IMDb menores que 1:',
      (datos_sin_corregir.puntaje_imdb < PUNTAJE_IMDB_MIN).sum(),
      '| mayores que 10:',
      (datos_sin_corregir.puntaje_imdb > PUNTAJE_IMDB_MAX).sum())
print('Cantidad de votos en cero o negativa:',
      (datos_sin_corregir.cantidad_votos <= 0).sum())
print('Años fuera de 1980–2020:',
      (~datos_sin_corregir.anio_estreno.between(
          ANIO_INICIAL_DATOS, ANIO_FINAL_DATOS)).sum())

print('\nEjemplos originales de valores imposibles:')
mascara_ejemplos = (reglas['recaudacion_mundial_usd']
                    | reglas['duracion_minutos'] | reglas['puntaje_imdb'])
display(datos_sin_corregir.loc[mascara_ejemplos,
    ['titulo','duracion_minutos','puntaje_imdb','recaudacion_mundial_usd']]
    .head(8))

Observaciones: el presupuesto concentra la mayor parte de los faltantes. Antes de decidir cómo tratarlo hay que comprobar si falta de manera similar en décadas, países y géneros (punto 1.9).

## 1.5 Valores imposibles y decisiones de limpieza

Revisamos rangos numéricos. El presupuesto debe ser positivo; la recaudación no puede ser negativa; IMDb va de 1 a 10. En duración, los valores observados de 0 y 999 minutos son inválidos; conservamos las películas de 357 y 366 minutos porque una duración extraordinaria no implica un error. Marcamos los valores inválidos como faltantes en la copia `datos`: imputarlos inventaría información. La tabla informa el porcentaje modificado y los faltantes antes/después de cada cambio.

In [ ]:
PRESUPUESTO_BAJO_MAX = 1_000_000
RECAUDACION_ALTA_MIN = 20_000_000
MULTIPLO_EXTREMO_MIN = 100
MULTIPLO_MODERADO_MAX = 20

# Cociente diagnóstico de dos columnas originales; todavía no es el ROI de la etapa 2.
datos['recaudacion_sobre_presupuesto'] = (
    datos.recaudacion_mundial_usd / datos.presupuesto_usd)

datos['escala_dudosa'] = (
    (datos.presupuesto_usd > 0)
    & (datos.presupuesto_usd < PRESUPUESTO_BAJO_MAX)
    & (datos.recaudacion_mundial_usd >= RECAUDACION_ALTA_MIN)
    & (datos.recaudacion_sobre_presupuesto > MULTIPLO_EXTREMO_MIN)
)

print('Candidatas a escala dudosa:',datos.escala_dudosa.sum(),
      f'({datos.escala_dudosa.mean():.2%} de la base depurada)')

# Primera tabla: presupuestos pequeños con recaudación muy alta.
print('Presupuestos pequeños con recaudación muy alta:')
display(datos.loc[datos.escala_dudosa,
                  ['titulo','anio_estreno','presupuesto_usd',
                   'recaudacion_mundial_usd','recaudacion_sobre_presupuesto']]
        .sort_values('recaudacion_sobre_presupuesto',ascending=False)
        .head(8).round(1))

# Segunda tabla: presupuestos pequeños con una relación más moderada.
print('Presupuestos pequeños con una relación más moderada:')
display(datos.loc[(datos.presupuesto_usd > 0)
                  & (datos.presupuesto_usd < PRESUPUESTO_BAJO_MAX)
                  & datos.recaudacion_sobre_presupuesto.between(
                      0, MULTIPLO_MODERADO_MAX),
                  ['titulo','presupuesto_usd','recaudacion_mundial_usd',
                   'recaudacion_sobre_presupuesto']].head(6).round(1))

**Decisión e impacto:** se marcaron como faltantes solo las celdas imposibles; se conservaron las 7668 películas. Se modificaron 48 presupuestos (0,63 % de la base), 11 recaudaciones (0,14 %), 34 duraciones (0,44 %) y 22 puntajes IMDb (0,29 %). La tabla muestra el porcentaje faltante de cada columna antes y después; por ejemplo, presupuesto pasó de 28,31 % a 28,94 %. No se imputó ningún valor porque no hay evidencia para inventarlo. El cero en recaudación se conserva porque puede representar ausencia de ingresos en salas; un valor negativo sí es imposible.

## 1.6 Presupuestos con escala dudosa

Un presupuesto muy pequeño no demuestra por sí solo un error. Lo cruzamos con la recaudación: presupuestos positivos menores a USD 1 millón, recaudación de al menos USD 20 millones y un múltiplo mayor que 100 forman un **filtro diagnóstico**. Estos límites son criterios de revisión elegidos para buscar casos extremos, no reglas del negocio ni prueba de error: una película barata también puede ser un éxito real.

In [ ]:
categorias_clave = ['genero','clasificacion','pais_origen','es_secuela',
                    'productora']
# Guardamos los valores previos para medir los cambios.
antes_categorias = datos[categorias_clave].copy()

# Decisión: unificamos variantes para no dividir una categoría al agrupar.
mapa_genero = {'accion':'Acción','biografia':'Biografía',
               'animacion':'Animación','ciencia ficcion':'Ciencia ficción'}
datos['genero'] = (datos.genero.str.strip().str.lower()
                   .replace(mapa_genero).str.capitalize())

mapa_clasificacion = {
    'atp':'ATP','apta todo publico':'ATP',
    'supervision parental':'Supervisión parental',
    'supervisión parental':'Supervisión parental',
    'mayores de 13':'Mayores de 13','mayores de 17':'Mayores de 17',
    'solo adultos':'Solo adultos','sin clasificar':'Sin clasificar'
}
datos['clasificacion'] = (datos.clasificacion.str.strip().str.lower()
                           .replace(mapa_clasificacion).fillna('No informado'))

datos['pais_origen'] = datos.pais_origen.str.strip().replace({
    'EE.UU.':'Estados Unidos','UK':'Reino Unido',
    'Canada':'Canadá','Japon':'Japón','Espania':'España','Mexico':'México'
})

datos['es_secuela'] = datos.es_secuela.str.strip()
datos['productora'] = datos.productora.str.strip().str.title()

# Contamos las filas y categorías afectadas en cada columna.
filas_categorias=[]
for columna in categorias_clave:
    modificadas=(antes_categorias[columna].fillna('') !=
                 datos[columna].fillna('')).sum()
    filas_categorias.append({
        'columna':columna,
        'categorias_antes':antes_categorias[columna].nunique(dropna=False),
        'filas_modificadas':int(modificadas),
        'base_afectada_pct':round(100*modificadas/len(datos),2),
        'no_nulos_antes_pct':round(100*antes_categorias[columna].notna().mean(),2),
        'no_nulos_despues_pct':round(100*datos[columna].notna().mean(),2),
        'categorias_despues':datos[columna].nunique(dropna=False)
    })

print('Cambios en las variables categóricas:')
display(pd.DataFrame(filas_categorias))

Decisión: no multiplicamos ni borramos presupuestos por esta regla. Las dos tablas muestran por qué el cruce con recaudación ayuda a localizar candidatos, pero no resuelve todos los casos. `escala_dudosa` queda como marca para revisar antes de estimar inversiones.

## 1.7 Normalizar categorías

Unificamos espacios, mayúsculas y variantes equivalentes en las categorías. También revisamos los nombres: director, guionista y protagonista no tienen variantes de espacios o mayúsculas en esta base; productora sí. Conservamos países históricos como `Alemania Occidental` o `Yugoslavia` porque no son simples errores de escritura. `es_secuela` ya presenta dos categorías coherentes.

In [ ]:
ANIO_COBERTURA_REGULAR_INICIO = 1985
ANIO_COBERTURA_REGULAR_FIN = 2019

cobertura = datos.groupby('anio_estreno').agg(
    peliculas=('id_pelicula','size'),
    con_presupuesto=('presupuesto_usd','count'),
    con_recaudacion=('recaudacion_mundial_usd','count')).reset_index()
cobertura['sin_recaudacion_pct'] = (
    100*(1-cobertura.con_recaudacion/cobertura.peliculas)).round(1)

display(pd.concat([cobertura.head(6),cobertura.tail(6)]))

print('Años presentes:',cobertura.anio_estreno.nunique(),
      '| rango:',cobertura.anio_estreno.min(),'-',cobertura.anio_estreno.max())
print(f'Películas por año entre {ANIO_COBERTURA_REGULAR_INICIO} y '
      f'{ANIO_COBERTURA_REGULAR_FIN}: mínimo',
      cobertura.loc[cobertura.anio_estreno.between(
          ANIO_COBERTURA_REGULAR_INICIO, ANIO_COBERTURA_REGULAR_FIN),
          'peliculas'].min(),
      'y máximo',
      cobertura.loc[cobertura.anio_estreno.between(
          ANIO_COBERTURA_REGULAR_INICIO, ANIO_COBERTURA_REGULAR_FIN),
          'peliculas'].max())

# Comprobamos si el año declarado coincide con la fecha de estreno.
anio_distinto = ((datos.fecha_convertida.dt.year != datos.anio_estreno)
                 & datos.fecha_convertida.notna())
print('Año declarado distinto del año de fecha:',anio_distinto.sum(),
      f'({anio_distinto.mean():.1%})')
print('Años de películas sin IPC:',
      len(set(datos.anio_estreno)-set(ipc.anio)))

Decisión: en 1.7 no se elimina ninguna película; se cambian textos en la copia `datos` para que una misma categoría no quede dividida al agrupar. Se corrigieron 696 filas de género (9,08 %; por ejemplo, `ACCION` → `Acción`), 961 de clasificación (12,53 %; `Apta todo publico` → `ATP`), 764 de país (9,96 %; `UK` → `Reino Unido`) y 2060 de productora (26,86 %; espacios y mayúsculas). `es_secuela` no necesitó cambios.

La tabla muestra el número de categorías y el porcentaje de valores no nulos antes y después de cada cambio, junto con el porcentaje de filas modificadas. Los faltantes de clasificación pasan a la etiqueta `No informado`: ya no son nulos en pandas, pero siguen sin aportar una clasificación real. Los de país y productora siguen faltantes. No se unifican nombres diferentes de productoras sin evidencia de que sean la misma empresa. Los porcentajes son por columna y pueden incluir una misma película más de una vez.

## 1.8 Cobertura temporal

Contamos títulos y valores económicos disponibles por año. Se compara la fecha convertida con `anio_estreno`, pero no se reemplaza automáticamente el año declarado: las fuentes pueden registrar estrenos distintos.

In [ ]:
ANIOS_POR_DECADA = 10

# Marcamos los faltantes para comparar grupos sin borrar ni imputar filas.
datos['sin_presupuesto'] = datos.presupuesto_usd.isna()
datos['decada'] = (datos.anio_estreno // ANIOS_POR_DECADA) * ANIOS_POR_DECADA

def tabla_faltantes(columna):
    tabla = datos.groupby(columna,dropna=False).agg(
        peliculas=('id_pelicula','size'),
        sin_presupuesto_pct=('sin_presupuesto','mean')).reset_index()
    tabla['sin_presupuesto_pct'] = (100*tabla.sin_presupuesto_pct).round(1)
    return tabla.sort_values('peliculas',ascending=False)

print('Por década:')
display(tabla_faltantes('decada'))

print('Por género:')
display(tabla_faltantes('genero'))

print('Por país:')
display(tabla_faltantes('pais_origen'))

sin_presupuesto = int(datos.sin_presupuesto.sum())
con_ambos = int((datos.presupuesto_usd.notna()
                 & datos.recaudacion_mundial_usd.notna()).sum())
print('Sin presupuesto:',sin_presupuesto,
      f'({sin_presupuesto/len(datos):.1%} de {len(datos)} películas)')
print('Con presupuesto y recaudación para análisis futuro:',con_ambos,
      f'({con_ambos/len(datos):.1%} de la base depurada)')

Observaciones: 2020 contiene muchos menos títulos y menos recaudaciones informadas que los años inmediatamente anteriores; no se debe interpretar su mediana como una variación anual comparable. Los primeros años también tienen menos títulos. Se conserva `anio_estreno` como referencia para análisis posteriores y se documenta la discrepancia con la fecha.

## 1.9 ¿Falta presupuesto al azar?

Comparamos la proporción sin presupuesto por década, género y país normalizados. La fila de 2020 representa un solo año, no una década completa. Mostramos el tamaño de cada grupo; las tasas de países y géneros con pocas películas deben interpretarse con cautela.

In [ ]:
ANIO_BASE_IPC = 2020
USD_POR_MILLON = 1_000_000

# Cada año del IPC debe aportar un solo valor al cruce.
ipc_para_cruce = ipc.rename(columns={'anio':'anio_estreno'})
print('Años repetidos en IPC:', ipc_para_cruce.anio_estreno.duplicated().sum())

filas_antes_ipc = len(datos)
datos = datos.merge(ipc_para_cruce, on='anio_estreno', how='left')
print('Películas antes y después del cruce:', filas_antes_ipc, 'y', len(datos))
print('Películas sin IPC:', datos.ipc_eeuu.isna().sum())

ipc_base = ipc.loc[ipc.anio == ANIO_BASE_IPC, 'ipc_eeuu'].iloc[0]
datos['factor_a_2020'] = ipc_base / datos.ipc_eeuu
datos['presupuesto_2020_usd'] = (
    datos.presupuesto_usd * datos.factor_a_2020)
datos['recaudacion_2020_usd'] = (
    datos.recaudacion_mundial_usd * datos.factor_a_2020)

print(f'IPC de referencia, año {ANIO_BASE_IPC}:', ipc_base)
print('Presupuestos disponibles antes y después:',
      datos.presupuesto_usd.notna().sum(), 'y',
      datos.presupuesto_2020_usd.notna().sum())
print('Recaudaciones disponibles antes y después:',
      datos.recaudacion_mundial_usd.notna().sum(), 'y',
      datos.recaudacion_2020_usd.notna().sum())

# Mostramos importes en millones para que la comparación sea legible.
muestra_1980 = datos.loc[(datos.anio_estreno == ANIO_INICIAL_DATOS)
                        & datos.presupuesto_usd.notna()
                        & datos.recaudacion_mundial_usd.notna()].head(3).copy()
muestra_1980['presupuesto_corriente_millones'] = (
    muestra_1980.presupuesto_usd / USD_POR_MILLON)
muestra_1980['presupuesto_2020_millones'] = (
    muestra_1980.presupuesto_2020_usd / USD_POR_MILLON)
muestra_1980['recaudacion_corriente_millones'] = (
    muestra_1980.recaudacion_mundial_usd / USD_POR_MILLON)
muestra_1980['recaudacion_2020_millones'] = (
    muestra_1980.recaudacion_2020_usd / USD_POR_MILLON)

print('Ejemplos de', ANIO_INICIAL_DATOS, ': importes en millones de USD:')
display(muestra_1980[['titulo','factor_a_2020',
                     'presupuesto_corriente_millones',
                     'presupuesto_2020_millones',
                     'recaudacion_corriente_millones',
                     'recaudacion_2020_millones']].round(2))

**Decisión de la etapa 1:** conservamos las 2219 películas sin presupuesto (28,9 % de la base depurada) y no imputamos valores sin evidencia. El faltante varía: 43,9 % en los años 1980 frente a 18,0 % en los 2000; 42,5 % en drama frente a 17,0 % en acción. Por eso, las 5383 películas con presupuesto y recaudación (70,2 % de la base) no representan por igual a todos los grupos. En etapas posteriores, los retornos se calcularán solo con ambas cifras disponibles y se aclarará este sesgo. Las demás películas siguen disponibles para otros análisis.

**Fin de la etapa 1.** La etapa 2 usa esta copia depurada para crear importes comparables y medidas de retorno; la etapa 3 analiza crecimiento, retorno y riesgo.

# Etapa 2: creación de variables

## 2.1 Presupuesto y recaudación en dólares de 2020

Los importes originales están en dólares corrientes de anio_estreno. Unimos el IPC anual a cada película y usamos 2020, el último año disponible, como referencia. La operación es **importe del año × IPC de 2020 / IPC de ese año**. Así expresamos importes de años distintos con el poder de compra de 2020. Usamos anio_estreno porque es el año al que la consigna asigna los importes; las discrepancias con fecha_estreno ya quedaron documentadas en 1.8.

In [ ]:
# Supuestos fijos del caso; se nombran como constantes, como en Clean Code.
PORCION_BOLETERIA_ESTUDIO = 0.50
MARKETING_SOBRE_PRESUPUESTO = 0.50
UMBRAL_RECUPERO = 2.5

# El mismo factor de IPC en numerador y denominador deja igual el múltiplo.
datos['multiplo_boleteria'] = (
    datos.recaudacion_2020_usd / datos.presupuesto_2020_usd)
ingreso_estudio_estimado = (
    datos.recaudacion_2020_usd * PORCION_BOLETERIA_ESTUDIO)
costo_total_estimado = (
    datos.presupuesto_2020_usd * (1 + MARKETING_SOBRE_PRESUPUESTO))
datos['roi_salas_estimado'] = (
    (ingreso_estudio_estimado - costo_total_estimado) / costo_total_estimado)

# Sin las dos cifras, no asignamos un resultado de recuperación.
con_ambos = (datos.presupuesto_2020_usd.notna()
             & datos.recaudacion_2020_usd.notna())
datos['recupero_2_5x'] = 'Sin datos'
datos.loc[con_ambos, 'recupero_2_5x'] = 'No'
datos.loc[con_ambos & (datos.multiplo_boleteria >= UMBRAL_RECUPERO),
          'recupero_2_5x'] = 'Sí'

print('ROI calculable:', datos.roi_salas_estimado.notna().sum(),
      f'({datos.roi_salas_estimado.notna().mean():.1%} de la base)')
print('Con escala de presupuesto dudosa entre los ROI calculables:',
      (con_ambos & datos.escala_dudosa).sum())
print('Películas por resultado de la regla 2,5×:')
display(datos.recupero_2_5x.value_counts().rename('peliculas').to_frame())

columnas_ejemplo = ['titulo','anio_estreno','multiplo_boleteria',
                    'roi_salas_estimado','recupero_2_5x']
ejemplos_roi = pd.concat([
    datos.loc[datos.recupero_2_5x == 'Sí', columnas_ejemplo].head(2),
    datos.loc[datos.recupero_2_5x == 'No', columnas_ejemplo].head(2),
    datos.loc[datos.recupero_2_5x == 'Sin datos', columnas_ejemplo].head(2)
])
print('Ejemplos de las variables nuevas:')
display(ejemplos_roi.round(2))

Observaciones: el factor de 2020 es 1; en 1980 es aproximadamente 3,14 (258,8 / 82,4). Multiplicar por ese factor cambia la unidad de comparación, no el dinero que la película recibió. El cruce mantuvo las 7668 películas y todas encontraron IPC. Los 5449 presupuestos y las 7468 recaudaciones disponibles siguen disponibles después de la conversión; los faltantes originales no se imputan.

## 2.2 Retorno estimado y recuperación de la inversión

La recaudación bruta no es ganancia. Para estimar un **ROI de salas**, usamos dos supuestos dados en la consigna: el exhibidor retiene aproximadamente la mitad de la boletería y el marketing y la distribución cuestan cerca del 50 % del presupuesto de producción (el extremo inferior del rango indicado). Los definimos como constantes con nombre. El ingreso estimado para el estudio es la recaudación multiplicada por su parte de boletería; el costo estimado es el presupuesto más marketing. Calculamos **(ingreso estimado − costo estimado) / costo estimado**, usando ambos importes en dólares de 2020. Es una aproximación, no el retorno contractual del fondo.

Por separado, recupero_2_5x aplica la regla comercial pedida: **Sí** cuando la recaudación mundial alcanza 2,5 veces el presupuesto; **No** cuando no llega, y **Sin datos** si falta alguna cifra. Esta regla de 2,5× y el ROI estimado usan criterios distintos, por lo que una película cercana al umbral puede quedar clasificada como recuperada y tener un ROI de salas estimado levemente negativo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Paleta usada en las clases: azul para la serie base y dorado para destacar.
AZUL_GRAFICO = '#11224D'
AZUL_CLARO_GRAFICO = '#6699CC'
DORADO_GRAFICO = '#D4AF37'
DECADA_COMPARACION_INICIO = 1990

evolucion = datos.groupby('anio_estreno').agg(
    peliculas=('id_pelicula','size'),
    con_recaudacion=('recaudacion_mundial_usd','count'),
    nominal_mediana=('recaudacion_mundial_usd','median'),
    real_mediana=('recaudacion_2020_usd','median')).reset_index()
evolucion['sin_recaudacion_pct'] = (
    100 * (1 - evolucion.con_recaudacion / evolucion.peliculas)).round(1)
evolucion['nominal_millones'] = evolucion.nominal_mediana / USD_POR_MILLON
evolucion['real_millones'] = evolucion.real_mediana / USD_POR_MILLON

serie_regular = evolucion.loc[evolucion.anio_estreno.between(
    ANIO_COBERTURA_REGULAR_INICIO, ANIO_COBERTURA_REGULAR_FIN)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(serie_regular.anio_estreno, serie_regular.nominal_millones,
        label='Dólares corrientes', color=AZUL_GRAFICO)
ax.plot(serie_regular.anio_estreno, serie_regular.real_millones,
        label='Dólares de 2020', color=DORADO_GRAFICO)
ax.set_title('La recaudación crece menos al corregir por inflación', loc='left')
ax.set_xlabel('Año de estreno')
ax.set_ylabel('USD millones por película')
ax.set_ylim(0, None)
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

# Comparamos décadas completas para resumir la tendencia de las dos series.
decadas_completas = datos.loc[datos.anio_estreno.between(
    DECADA_COMPARACION_INICIO, ANIO_COBERTURA_REGULAR_FIN)]
resumen_decadas = decadas_completas.groupby('decada').agg(
    peliculas=('id_pelicula','size'),
    nominal_mediana=('recaudacion_mundial_usd','median'),
    real_mediana=('recaudacion_2020_usd','median')).reset_index()
resumen_decadas['nominal_mediana_millones'] = (
    resumen_decadas.nominal_mediana / USD_POR_MILLON)
resumen_decadas['real_mediana_millones'] = (
    resumen_decadas.real_mediana / USD_POR_MILLON)

print('Recaudación mediana por década completa, en USD millones:')
display(resumen_decadas[
    ['decada','peliculas','nominal_mediana_millones',
     'real_mediana_millones']].round(2))
print('Cobertura del último año, excluido del gráfico:')
display(evolucion.loc[evolucion.anio_estreno == ANIO_BASE_IPC,
                      ['anio_estreno','peliculas','con_recaudacion',
                       'sin_recaudacion_pct']])

**Decisión e impacto:** se conservan las 7668 películas. Hay 5383 ROI calculables (70,2 %); en las otras 2285 (29,8 %) falta al menos una cifra y el indicador de recuperación dice `Sin datos`. Entre las calculables, 31 tienen `escala_dudosa`: el cálculo existe, pero no debe usarse como evidencia firme hasta verificar el presupuesto.

## 2.3 Variables adicionales

`factor_a_2020` muestra cuánto se ajustó cada importe por inflación y `multiplo_boleteria` muestra cuántas veces la recaudación bruta cubre el presupuesto. Ambas ayudan a interpretar las comparaciones y a revisar casos extremos en las etapas siguientes.

# Etapa 3: análisis de negocio

## 3.1 ¿Creció la industria?

Comparamos la recaudación mediana por año en dólares corrientes y en dólares de 2020. Usamos la mediana para que unas pocas películas enormes no definan la serie. El gráfico se concentra en 1985–2019, período con 200 títulos por año en esta base; los primeros años tienen menos títulos y 2020 está incompleto. La serie mide lo que ocurre en la muestra entregada, no toda la producción mundial.

In [ ]:
PRESUPUESTO_MIN_ANALISIS = 1_000_000
MIN_PELICULAS_COMPARABLES = 30

con_roi = datos.loc[datos.roi_salas_estimado.notna()].copy()
con_roi_hasta_2019 = con_roi.loc[
    con_roi.anio_estreno <= ANIO_COBERTURA_REGULAR_FIN].copy()
universo_negocio = con_roi_hasta_2019.loc[
    con_roi_hasta_2019.presupuesto_usd >= PRESUPUESTO_MIN_ANALISIS].copy()
universo_negocio['roi_pct'] = 100 * universo_negocio.roi_salas_estimado
universo_negocio['fracaso_25'] = universo_negocio.recupero_2_5x == 'No'

print('ROI disponibles:', len(con_roi))
print('Excluidas por 2020 incompleto:',
      len(con_roi) - len(con_roi_hasta_2019))
print('Excluidas por presupuesto menor a USD 1 millón:',
      len(con_roi_hasta_2019) - len(universo_negocio),
      f'({(len(con_roi_hasta_2019) - len(universo_negocio)) / len(con_roi_hasta_2019):.1%}'
      ' de los ROI hasta 2019)')
print('Universo principal:', len(universo_negocio),
      f'({len(universo_negocio) / len(datos):.1%} de la base depurada)')

resumen_genero = universo_negocio.groupby('genero').agg(
    peliculas=('id_pelicula','size'),
    roi_medio_pct=('roi_pct','mean'),
    roi_mediano_pct=('roi_pct','median')).reset_index()
resumen_genero['diferencia_pp'] = (
    resumen_genero.roi_medio_pct - resumen_genero.roi_mediano_pct)
print('ROI por género, en porcentaje:')
display(resumen_genero.sort_values('peliculas', ascending=False).round(1))

resumen_secuela = universo_negocio.groupby('es_secuela').agg(
    peliculas=('id_pelicula','size'),
    roi_medio_pct=('roi_pct','mean'),
    roi_mediano_pct=('roi_pct','median')).reset_index()
print('ROI por franquicia, en porcentaje:')
display(resumen_secuela.round(1))

# El efecto del filtro se observa al comparar las mismas medias por género.
media_todos = con_roi_hasta_2019.groupby(
    'genero').roi_salas_estimado.mean() * 100
media_principal = universo_negocio.groupby(
    'genero').roi_salas_estimado.mean() * 100
sensibilidad = pd.DataFrame({
    'media_con_todos_pct':media_todos,
    'media_principal_pct':media_principal
})
print('Efecto del filtro sobre algunas medias por género:')
display(sensibilidad.loc[['Aventura','Comedia','Terror']].round(1))

**Conclusión 3.1:** en esta base, la película mediana de los años 2010 recaudó más que la de los años 1990. En dólares corrientes, la recaudación mediana pasó de USD 12,89 a 41,68 millones: parece haberse multiplicado por 3,2. Al expresar ambas décadas en dólares de 2020, pasó de USD 22,15 a 45,81 millones: creció cerca de 2,1 veces. **Parte del aumento se explica por la inflación, pero también hubo crecimiento después de corregirla.** No usamos 2020 para cerrar la comparación porque solo tiene 25 películas y al 40 % le falta la recaudación.

## 3.2 ¿Cuánto rinde cada tipo de película?

Calculamos el ROI medio y mediano por género y repetimos el análisis por franquicia (es_secuela). El presupuesto de escala dudosa detectado en 1.6 puede crear retornos extremos falsos incluso cuando la recaudación no supera USD 20 millones. Para la comparación principal usamos películas con ambas cifras, estreno hasta 2019 y presupuesto informado de al menos USD 1 millón. Es un **filtro de análisis**, no un borrado ni una corrección del archivo: excluye también éxitos genuinos muy baratos. Mostramos su efecto sobre las medias para que la conclusión no dependa de ocultarlo.

In [ ]:
# Comparamos media y mediana solo en géneros con suficiente muestra.
ANCHO_BARRA = 0.36
MARGEN_EJE_PCT = 0.10
generos_comparacion = resumen_genero.loc[
    resumen_genero.peliculas >= MIN_PELICULAS_COMPARABLES].sort_values(
        'roi_mediano_pct')
posiciones = np.arange(len(generos_comparacion))

fig, ax = plt.subplots(figsize=(11, 6))
barras_media = ax.barh(
    posiciones + ANCHO_BARRA / 2, generos_comparacion.roi_medio_pct,
    height=ANCHO_BARRA, color=DORADO_GRAFICO, label='Media')
barras_mediana = ax.barh(
    posiciones - ANCHO_BARRA / 2, generos_comparacion.roi_mediano_pct,
    height=ANCHO_BARRA, color=AZUL_GRAFICO, label='Mediana')

ax.set_yticks(posiciones)
ax.set_yticklabels(generos_comparacion.genero)
roi_min = min(generos_comparacion.roi_medio_pct.min(),
              generos_comparacion.roi_mediano_pct.min())
roi_max = max(generos_comparacion.roi_medio_pct.max(),
              generos_comparacion.roi_mediano_pct.max())
margen_eje = (roi_max - roi_min) * MARGEN_EJE_PCT
ax.set_xlim(roi_min - margen_eje, roi_max + margen_eje)
ax.bar_label(barras_media, fmt='%.1f%%', padding=3, fontsize=8)
ax.bar_label(barras_mediana, fmt='%.1f%%', padding=3, fontsize=8)
ax.set_title('En terror, la media del ROI supera ampliamente a la mediana',
             loc='left')
ax.set_xlabel('ROI estimado de salas (%)')
ax.set_ylabel('Género')
ax.legend(frameon=False)
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print('Géneros mostrados:', len(generos_comparacion),
      '| mínimo por género:', MIN_PELICULAS_COMPARABLES)

In [ ]:
print('Películas con mayor ROI en el universo principal:')
display(universo_negocio.sort_values('roi_salas_estimado', ascending=False)
        [['titulo','genero','presupuesto_usd',
          'recaudacion_mundial_usd','roi_pct']].head(8).round(1))

# Contrastamos importes pequeños: un múltiplo enorme no prueba un error.
casos_a_revisar = datos.loc[datos.titulo.isin(
    ['Platoon','Paranormal Activity','The Devil Inside']),
    ['titulo','anio_estreno','presupuesto_usd',
     'recaudacion_mundial_usd','multiplo_boleteria','escala_dudosa']]
print('Casos que ayudan a distinguir errores de éxitos reales:')
display(casos_a_revisar.round(1))

**Conclusión del gráfico:** en terror, el ROI medio es 151,6 % y el mediano apenas 0,2 %: unos pocos éxitos extremos elevan mucho el promedio. Las barras están ordenadas por mediana para comparar el resultado típico, y los grupos pequeños quedan fuera de esta visualización.

In [ ]:
# Correlación de Pearson en el mismo universo usado para comparar retornos.
variables_correlacion = ['puntaje_imdb','roi_salas_estimado']
matriz_correlacion = universo_negocio[variables_correlacion].corr()
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(matriz_correlacion, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True,
            cbar_kws={'label':'Correlación'}, ax=ax)
correlacion_imdb_roi = matriz_correlacion.loc[
    'puntaje_imdb','roi_salas_estimado']
ax.set_title(f'IMDb y ROI tienen correlación débil ({correlacion_imdb_roi:.2f})',
             loc='left')
fig.tight_layout()
plt.show()

print('Películas con puntaje IMDb y ROI:',
      universo_negocio[variables_correlacion].dropna().shape[0])
print('Correlación IMDb–ROI:', round(correlacion_imdb_roi, 3))

**Lectura de extremos:** las medias pueden quedar muy por encima de las medianas porque unas pocas películas tienen retornos enormes. En terror, por ejemplo, The Devil Inside y Saw aparecen entre los máximos del universo principal. La base registra USD 1 millón de presupuesto para The Devil Inside y [The Numbers](https://www.the-numbers.com/movie/Devil-Inside-The) informa el mismo orden de magnitud: es un éxito extremo plausible, no una cifra que debamos cambiar solo por ser grande.

El filtro de escala de 1.6 también contiene casos distintos. La base registra USD 6000 para Platoon, mientras que una [crónica de la época](https://time.com/3834671/platoon-oliver-stone-charlie-sheen-movie-review/) informa USD 6 millones: hay evidencia de un error de escala. En cambio, [Paramount](https://ir.paramount.com/static-files/f55db0fb-b0ca-455f-a36a-bc1f5dc03368) confirma el presupuesto de USD 15.000 de Paranormal Activity: su múltiplo excepcional refleja un caso real de muy bajo presupuesto. No corregimos masivamente los demás casos sin verificar cada fuente. El universo principal excluye provisionalmente los presupuestos inferiores a USD 1 millón y la comparación de medias muestra cuánto cambia la lectura.

In [ ]:
FRACCION_PEOR_DECIL = 0.10

def retorno_peor_decil(serie):
    limite = serie.quantile(FRACCION_PEOR_DECIL)
    return serie.loc[serie <= limite].mean()

def tabla_riesgo(columna):
    tabla = universo_negocio.groupby(columna).agg(
        peliculas=('id_pelicula','size'),
        roi_mediano_pct=('roi_pct','median'),
        fracaso_pct=('fracaso_25','mean'),
        roi_peor_decil_pct=('roi_pct',retorno_peor_decil)).reset_index()
    tabla['fracaso_pct'] = 100 * tabla.fracaso_pct
    return tabla.sort_values('peliculas', ascending=False)

riesgo_genero = tabla_riesgo('genero')
riesgo_secuela = tabla_riesgo('es_secuela')
print('Riesgo por género (ROI y fracaso en porcentaje):')
display(riesgo_genero.round(1))
print('Riesgo por franquicia (ROI y fracaso en porcentaje):')
display(riesgo_secuela.round(1))

# Aplicamos el mismo criterio de dominancia a las dos categorías de franquicia.
secuela_si = riesgo_secuela.loc[
    riesgo_secuela.es_secuela == 'Si'].iloc[0]
secuela_no = riesgo_secuela.loc[
    riesgo_secuela.es_secuela == 'No'].iloc[0]
print('Dominancia por franquicia:')
if (secuela_si.roi_mediano_pct > secuela_no.roi_mediano_pct
        and secuela_si.fracaso_pct < secuela_no.fracaso_pct):
    print('Las secuelas dominan a las no secuelas en mediana y fracaso.')
else:
    print('No hay dominancia de secuelas sobre no secuelas.')

# Dominancia compara mediana y fracaso; la propia fila no se domina a sí misma.
comparables = riesgo_genero.loc[
    riesgo_genero.peliculas >= MIN_PELICULAS_COMPARABLES].reset_index(drop=True).copy()
dominador_ejemplo = []
for i in range(len(comparables)):
    actual = comparables.iloc[i]
    mejor_retorno = comparables.roi_mediano_pct >= actual.roi_mediano_pct
    menor_fracaso = comparables.fracaso_pct <= actual.fracaso_pct
    ventaja_estricta = ((comparables.roi_mediano_pct > actual.roi_mediano_pct)
                        | (comparables.fracaso_pct < actual.fracaso_pct))
    nombres = comparables.loc[mejor_retorno & menor_fracaso & ventaja_estricta,
                             'genero']
    if len(nombres) > 0:
        dominador_ejemplo.append(nombres.iloc[0])
    else:
        dominador_ejemplo.append('Ninguno')

comparables['dominador_ejemplo'] = dominador_ejemplo
print('Dominancia entre géneros con al menos', MIN_PELICULAS_COMPARABLES,
      'películas:')
display(comparables.sort_values('roi_mediano_pct', ascending=False)
        [['genero','peliculas','roi_mediano_pct','fracaso_pct',
          'roi_peor_decil_pct','dominador_ejemplo']].round(1))
print('Géneros no dominados:',
      ', '.join(list(comparables.loc[
          comparables.dominador_ejemplo == 'Ninguno', 'genero'])))

**Conclusión 3.2:** en el universo principal, el ROI medio es 3,2 % y el mediano es −39,8 %. Entre los nueve géneros con al menos 30 películas, **solo terror tiene ROI mediano positivo, y es apenas 0,2 %**; animación queda cerca con −2,8 % y los otros siete tienen medianas más negativas. En terror la media llega a 151,6 % por unos pocos éxitos extremos, de modo que no describe a la película típica. Las secuelas tienen mediana de −1,5 % frente a −44,6 % en las no secuelas. El mapa de calor da una correlación IMDb–ROI de 0,17: es una asociación positiva débil, no una garantía ni una relación causal. El universo principal excluye presupuestos inferiores a USD 1 millón, incluidos éxitos verdaderos de bajo costo, por lo que no representa a todas las películas.

## 3.3 ¿Dónde está el riesgo?

Para cada género y para franquicia calculamos tres medidas: ROI mediano, proporción que **no recuperó la inversión según la regla 2,5×**, y retorno medio del 10 % de películas con peor ROI. Esta última cifra muestra cuánto se pierde en la cola desfavorable. La probabilidad de fracaso es una frecuencia histórica en esta muestra, no una probabilidad garantizada para un proyecto futuro.

Un género domina a otro si tiene **ROI mediano igual o mayor y proporción de fracaso igual o menor**, con al menos una ventaja estricta. Comparamos solo géneros con al menos 30 películas para reducir el peso de muestras diminutas; el peor decil se informa como medida adicional de riesgo.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = 'plotly_white'
GRIS_GRAFICO = '#BBBBBB'

figura_evolucion = go.Figure()
figura_evolucion.add_trace(go.Scatter(
    x=serie_regular.anio_estreno, y=serie_regular.nominal_millones,
    mode='lines', name='Dólares corrientes',
    line={'color':AZUL_GRAFICO,'width':2}))
figura_evolucion.add_trace(go.Scatter(
    x=serie_regular.anio_estreno, y=serie_regular.real_millones,
    mode='lines', name='Dólares de 2020',
    line={'color':DORADO_GRAFICO,'width':2}))
figura_evolucion.update_layout(
    title='La recaudación mediana creció menos al ajustar por inflación',
    xaxis_title='Año de estreno',
    yaxis_title='Recaudación mediana por película (USD millones)',
    hovermode='x unified', height=460)
figura_evolucion.update_yaxes(rangemode='tozero')
figura_evolucion.show()

print('Años mostrados:', len(serie_regular),
      '| desde', serie_regular.anio_estreno.min(),
      'hasta', serie_regular.anio_estreno.max())

**Conclusión 3.3:** entre los géneros con al menos 30 películas, terror y animación no están dominados. Terror tiene ROI mediano de 0,2 % y fracaso comercial de 42,8 %; animación tiene mediana de −2,8 % y fracaso de 41,9 %. Ninguno supera al otro en las dos medidas a la vez. Sus peores deciles promedian −92,8 % y −89,1 %, respectivamente: incluso estos géneros pueden sufrir pérdidas severas. Entre las categorías de franquicia, las secuelas dominan a las no secuelas: su mediana es −1,5 % frente a −44,6 % y su fracaso comercial 41,1 % frente a 64,5 %. Es una comparación histórica entre grupos de tamaño distinto, no prueba que hacer una secuela cause un mejor retorno. La etapa 5 deberá considerar también los presupuestos típicos y las películas excluidas.

# Etapa 4: visualización

Se presentan las cuatro opciones de la consigna, cada una con una conclusión breve inmediatamente debajo. La evolución y la comparación de géneros usan Plotly para consultar detalles; el histograma usa Matplotlib para señalar claramente media y mediana, y la dispersión usa transparencia y escala logarítmica para leer miles de puntos. Cuando el gráfico requiere ROI o presupuesto usamos el universo principal definido en 3.2: excluye 2020 y presupuestos inferiores a USD 1 millón. Los gráficos no modifican los datos.

## 4.1 Evolución nominal y real de la recaudación

Cada punto de las dos líneas representa la recaudación mediana de las películas de ese año. La distancia entre líneas muestra cuánto cambia la lectura al expresar la recaudación en dólares de 2020; al pasar el cursor se comparan los dos valores del mismo año. El período 1985–2019 evita los años con cobertura muy desigual indicados en 1.8.

In [ ]:
ROI_TOPE_VISUAL_PCT = 300
CANTIDAD_BINS_ROI = 40

roi_visual = universo_negocio[['roi_pct']].copy()
roi_visual['roi_grafico_pct'] = roi_visual.roi_pct.clip(
    upper=ROI_TOPE_VISUAL_PCT)
media_roi = roi_visual.roi_pct.mean()
mediana_roi = roi_visual.roi_pct.median()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(roi_visual.roi_grafico_pct, bins=CANTIDAD_BINS_ROI,
        color=AZUL_CLARO_GRAFICO, edgecolor='white')
ax.axvline(mediana_roi, color=AZUL_GRAFICO, linewidth=2.5,
           label=f'Mediana: {mediana_roi:.1f}%')
ax.axvline(media_roi, color=DORADO_GRAFICO, linewidth=2.5,
           linestyle='--', label=f'Media: {media_roi:.1f}%')
ax.set_title('La media positiva esconde pérdidas en la mayoría', loc='left')
ax.set_xlabel('ROI estimado de salas (%)')
ax.set_ylabel('Número de películas')
ax.set_ylim(0, None)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

print('Películas mostradas:', len(roi_visual))
print('Películas con ROI estimado negativo:',
      (roi_visual.roi_pct < 0).sum(),
      f'({(roi_visual.roi_pct < 0).mean():.1%})')
print('ROI mayores que', ROI_TOPE_VISUAL_PCT, '% agrupados visualmente:',
      (roi_visual.roi_pct > ROI_TOPE_VISUAL_PCT).sum())
print('ROI medio y mediano (%):',
      round(media_roi, 1), 'y', round(mediana_roi, 1))

**Conclusión:** la mediana de los años 2010 supera a la de los 1990 tanto en dólares corrientes como en dólares de 2020. El aumento parece de 3,2 veces sin ajustar y de 2,1 veces al corregir la inflación: parte del crecimiento nominal era un cambio en el valor del dólar.

## 4.2 Distribución del ROI estimado

Cada barra cuenta cuántas películas tienen un ROI dentro de un intervalo: a la izquierda de 0 % hay pérdidas estimadas y a la derecha, ganancias. La línea azul marca la **mediana** (el punto que divide las películas en dos mitades) y la dorada marca la **media** (la suma de los ROI dividida entre las películas); ambas se calculan con los valores originales. Si la mayor parte de las películas pierde pero la media queda positiva, unos pocos éxitos extremos están empujando el promedio y este no representa a la película típica.

Para poder ver la zona donde se concentra la mayoría, solo en el gráfico agrupamos los ROI superiores a 300 % en el último tramo. No usamos escala logarítmica porque el ROI puede ser negativo.

In [ ]:
generos_grafico = comparables.sort_values('roi_mediano_pct')

figura_generos = px.bar(
    generos_grafico, x='roi_mediano_pct', y='genero',
    orientation='h', text='roi_mediano_pct',
    hover_data=['peliculas','fracaso_pct','roi_peor_decil_pct'],
    color_discrete_sequence=[AZUL_GRAFICO],
    labels={'roi_mediano_pct':'ROI mediano estimado (%)',
            'genero':'Género',
            'peliculas':'Películas',
            'fracaso_pct':'Fracaso según 2,5× (%)',
            'roi_peor_decil_pct':'ROI medio del peor decil (%)'},
    title='Terror y animación encabezan el ROI mediano entre géneros comparables')
figura_generos.update_traces(texttemplate='%{text:.1f}%',
                             textposition='outside')
figura_generos.update_layout(height=530,
                             margin={'l':90,'r':80,'t':80,'b':60})
figura_generos.show()

print('Géneros mostrados:', len(generos_grafico),
      '| mínimo por género:', MIN_PELICULAS_COMPARABLES)

**Conclusión:** aunque el ROI medio es 3,2 %, el 69,0 % de las películas tiene ROI negativo y la mediana es −39,8 %: el promedio no describe a la película típica. Los 174 retornos superiores a 300 % elevan la media y se agrupan solo en la visualización, sin cambiar los datos.

## 4.3 Retorno mediano por género

Cada barra muestra el ROI mediano de un género: las barras a la izquierda de 0 % indican que su película típica tuvo un retorno estimado negativo. Ordenamos por valor, no por nombre, y mostramos solo géneros con al menos 30 películas para no destacar resultados basados en muy pocos casos. Al pasar el cursor se ven el tamaño del grupo, su frecuencia de fracaso según 2,5× y el ROI medio de su peor decil; la mediana sola no resume todo el riesgo.

In [ ]:
# La escala logarítmica requiere importes estrictamente positivos.
dispersion = universo_negocio.loc[
    (universo_negocio.presupuesto_2020_usd > 0)
    & (universo_negocio.recaudacion_2020_usd > 0)].copy()
dispersion['presupuesto_millones'] = (
    dispersion.presupuesto_2020_usd / USD_POR_MILLON)
dispersion['recaudacion_millones'] = (
    dispersion.recaudacion_2020_usd / USD_POR_MILLON)

fig, ax = plt.subplots(figsize=(9, 6))
for resultado, color, etiqueta in [
    ('No', GRIS_GRAFICO, 'No recuperó 2,5×'),
    ('Sí', AZUL_GRAFICO, 'Recuperó 2,5×')
]:
    grupo = dispersion.loc[dispersion.recupero_2_5x == resultado]
    ax.scatter(grupo.presupuesto_millones, grupo.recaudacion_millones,
               s=12, alpha=0.25, color=color, label=etiqueta)

presupuesto_min = dispersion.presupuesto_millones.min()
presupuesto_max = dispersion.presupuesto_millones.max()
ax.plot([presupuesto_min, presupuesto_max],
        [UMBRAL_RECUPERO * presupuesto_min,
         UMBRAL_RECUPERO * presupuesto_max],
        color=DORADO_GRAFICO, linestyle='--', label='Umbral de 2,5×')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Presupuestos similares tienen recaudaciones muy distintas',
             loc='left')
ax.set_xlabel('Presupuesto (USD millones de 2020, escala log)')
ax.set_ylabel('Recaudación mundial (USD millones de 2020, escala log)')
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

print('Películas mostradas:', len(dispersion),
      '| excluidas por importe cero para la escala log:',
      len(universo_negocio) - len(dispersion))

**Conclusión:** entre los nueve géneros comparables, solo terror tiene una mediana positiva y es apenas 0,2 %; animación le sigue con −2,8 % y policial queda último con −56,2 %. Incluso los grupos mejor ubicados pueden perder, por lo que el orden se debe leer junto con el tamaño y el riesgo que aparecen al pasar el cursor.

## 4.4 Presupuesto frente a recaudación

Cada punto es una película: su posición horizontal muestra el presupuesto y la vertical, la recaudación mundial, ambos en millones de dólares de 2020. La línea dorada indica recaudación igual a 2,5 veces el presupuesto; los puntos azules están por encima de la regla de recuperación y los grises por debajo. Películas con presupuesto parecido pueden quedar a alturas muy distintas. Para revelar zonas densas sin que miles de puntos formen una mancha usamos puntos pequeños, transparencia y escalas logarítmicas; los importes iguales a cero no pueden ponerse en esa escala y se cuentan por separado.

In [ ]:
CORTES_PRESUPUESTO_M = [0, 10, 30, 70, np.inf]
NOMBRES_TRAMOS = ['Hasta 10', '10-30', '30-70', 'Más de 70']

portafolio_base = universo_negocio.copy()
portafolio_base['presupuesto_millones'] = (
    portafolio_base.presupuesto_2020_usd / USD_POR_MILLON)
portafolio_base['tramo_presupuesto'] = pd.cut(
    portafolio_base.presupuesto_millones,
    bins=CORTES_PRESUPUESTO_M, labels=NOMBRES_TRAMOS)

# Verificamos que los tramos propuestos tengan casos suficientes.
resumen_tramos = portafolio_base.groupby(
    'tramo_presupuesto', observed=True).agg(
        peliculas=('id_pelicula','size'),
        presupuesto_mediano_m=('presupuesto_millones','median')).reset_index()
print('Películas y presupuesto mediano por tramo:')
display(resumen_tramos.round(1))

# Cada fila define un grupo observable; la asignación se decide en 5.2.
grupos_elegidos = pd.DataFrame([
    {'grupo':'Acción, más de 70 M, secuela',
     'genero':'Acción', 'tramo':'Más de 70', 'es_secuela':'Si'},
    {'grupo':'Animación, más de 70 M, secuela',
     'genero':'Animación', 'tramo':'Más de 70', 'es_secuela':'Si'},
    {'grupo':'Acción, 30-70 M, secuela',
     'genero':'Acción', 'tramo':'30-70', 'es_secuela':'Si'},
    {'grupo':'Comedia, 30-70 M, secuela',
     'genero':'Comedia', 'tramo':'30-70', 'es_secuela':'Si'},
    {'grupo':'Terror, hasta 10 M, no secuela',
     'genero':'Terror', 'tramo':'Hasta 10', 'es_secuela':'No'}
])

filas_grupos = []
retornos_historicos = {}
for i in range(len(grupos_elegidos)):
    criterio = grupos_elegidos.iloc[i]
    historicas = portafolio_base.loc[
        (portafolio_base.genero == criterio['genero'])
        & (portafolio_base.tramo_presupuesto == criterio['tramo'])
        & (portafolio_base.es_secuela == criterio['es_secuela'])]
    retornos_historicos[criterio['grupo']] = (
        historicas.roi_salas_estimado.values)
    filas_grupos.append({
        'grupo':criterio['grupo'],
        'peliculas_historicas':len(historicas),
        'presupuesto_mediano_m':historicas.presupuesto_millones.median(),
        'roi_mediano_pct':100 * historicas.roi_salas_estimado.median(),
        'fracaso_pct':100 * historicas.fracaso_25.mean()
    })

resumen_grupos = pd.DataFrame(filas_grupos)
print('Cinco grupos propuestos; presupuestos en USD millones de 2020:')
display(resumen_grupos.round(1))
print('Mínimo de películas históricas por grupo:',
      resumen_grupos.peliculas_historicas.min())

# Comparamos los grupos intermedios con películas no secuela equivalentes.
comparadores = portafolio_base.loc[
    portafolio_base.genero.isin(['Acción', 'Comedia'])
    & (portafolio_base.tramo_presupuesto == '30-70')]
comparacion_secuela = comparadores.groupby(
    ['genero', 'es_secuela']).agg(
        peliculas=('id_pelicula', 'size'),
        roi_mediano_pct=('roi_salas_estimado', 'median'),
        fracaso_pct=('fracaso_25', 'mean')).reset_index()
comparacion_secuela['roi_mediano_pct'] = (
    100 * comparacion_secuela.roi_mediano_pct)
comparacion_secuela['fracaso_pct'] = (
    100 * comparacion_secuela.fracaso_pct)
print('Acción y comedia de 30–70 M: secuelas frente a no secuelas:')
display(comparacion_secuela.round(1))

**Conclusión:** películas con presupuestos parecidos alcanzan recaudaciones muy diferentes, por lo que un presupuesto mayor no asegura recuperar la inversión. La línea de 2,5× separa los resultados comerciales y la transparencia deja ver dónde se concentran las películas superpuestas.

# Etapa 5: tesis de inversión

La propuesta es un **portafolio ilustrativo sujeto a verificar presupuestos y contratos**. Usa los ROI de salas estimados en 2.2 y el universo principal de 3.2: películas con presupuesto y recaudación, estreno hasta 2019 y presupuesto de al menos USD 1 millón. No convierte la recaudación de salas en retorno contractual del fondo. Todos los presupuestos comparables están expresados en dólares de 2020, mientras que la línea de USD 500 millones del caso necesitaría actualizarse al poder de compra del año real de inversión.

## 5.1 Definir los grupos de películas

Definimos cinco grupos con criterios que se pueden comprobar en las columnas género, presupuesto de producción en dólares de 2020 y es_secuela. Elegimos **10, 30 y 70 millones como cortes redondos de planificación** para distinguir producciones pequeñas, intermedias y grandes; no son umbrales dados por la consigna ni saltos de ROI demostrados por los datos. Usamos pd.cut, visto en Feature Engineering, para mantener los mismos rangos al evaluar proyectos futuros. Comprobamos el tamaño de cada tramo antes de seleccionar los grupos. Como los resultados pueden variar con los cortes, al final de la etapa probamos valores cercanos. En este paso todavía no asignamos capital: primero verificamos tamaño histórico, presupuesto mediano, ROI mediano y fracaso comercial de cada grupo.

In [ ]:
CAPITAL_TOTAL_M = 500

asignacion = pd.DataFrame([
    {'grupo':'Acción, más de 70 M, secuela',
     'asignacion_m':170, 'proyectos':1},
    {'grupo':'Animación, más de 70 M, secuela',
     'asignacion_m':120, 'proyectos':1},
    {'grupo':'Acción, 30-70 M, secuela',
     'asignacion_m':100, 'proyectos':2},
    {'grupo':'Comedia, 30-70 M, secuela',
     'asignacion_m':47, 'proyectos':1},
    {'grupo':'Terror, hasta 10 M, no secuela',
     'asignacion_m':63, 'proyectos':12}
])

propuesta = resumen_grupos.merge(asignacion, on='grupo', how='left')
propuesta['capital_por_proyecto_m'] = (
    propuesta.asignacion_m / propuesta.proyectos)
propuesta['desvio_del_presupuesto_mediano_pct'] = (
    100 * (propuesta.capital_por_proyecto_m
           / propuesta.presupuesto_mediano_m - 1))
propuesta['participacion_pct'] = (
    100 * propuesta.asignacion_m / CAPITAL_TOTAL_M)

print('Asignación por grupo; capital en USD millones:')
display(propuesta.round(1))
print('Capital total asignado:', propuesta.asignacion_m.sum(),
      'M | proyectos:', propuesta.proyectos.sum())
print('Suma de presupuestos medianos de esos proyectos:',
      round((propuesta.proyectos * propuesta.presupuesto_mediano_m).sum(), 1),
      'M')
print('Grupos con capital por proyecto al menos igual al presupuesto mediano:',
      (propuesta.capital_por_proyecto_m
       >= propuesta.presupuesto_mediano_m).sum(), 'de', len(propuesta))

**Decisión 5.1:** elegimos grupos con criterios que se pueden comprobar y con al menos 33 películas históricas cada uno. La tabla anterior muestra por qué incluimos cada uno:

- **Acción, más de 70 M, secuela:** 141 casos y ROI mediano de +8,2 %. Aporta un grupo de grandes producciones con muestra relativamente amplia. Su presupuesto mediano de USD 163,8 M exige limitar la cantidad de proyectos.
- **Animación, más de 70 M, secuela:** ROI mediano de +41,0 % y fracaso comercial de 26,2 %, el menor de los cinco grupos. Sus 42 casos y su alto presupuesto aconsejan una asignación limitada.
- **Acción, 30–70 M, secuela:** ROI mediano de −8,6 %, pero mejor que el −51,7 % de la acción no secuela del mismo tramo; también registra menos fracasos (46,2 % frente a 71,9 %). La incluimos con cautela para sumar proyectos de costo intermedio.
- **Comedia, 30–70 M, secuela:** ROI mediano de −4,0 % frente al −47,8 % de la comedia no secuela del mismo tramo, y 36,4 % de fracasos frente a 65,9 %. Es una opción intermedia, pero sus 33 antecedentes y mediana negativa justifican una exposición pequeña.
- **Terror, hasta 10 M, no secuela:** ROI mediano de +44,6 % con presupuesto mediano de USD 5,2 M. Permite repartir el capital entre varios proyectos pequeños; la propuesta depende de encontrar suficientes proyectos comparables.

Estas diferencias históricas no demuestran que una secuela *cause* un mejor resultado. Además, seleccionar grupos por su desempeño pasado puede favorecer resultados que no se repitan.

## 5.2 Asignar USD 500 millones y cantidad de proyectos

Asignamos capital y proyectos a los grupos ya definidos. Para que el número de proyectos sea coherente con los datos, comparamos el capital por película con el presupuesto mediano de producción de su grupo. Los importes son montos de planificación redondeados, próximos a esos presupuestos típicos; no surgen de una optimización que garantice el mejor retorno. También comprobamos que sumen USD 500 M y contamos los proyectos. Los presupuestos comparables están en dólares de 2020 y no incluyen una estimación contractual del gasto de marketing.

In [ ]:
N_SIMULACIONES = 10_000
SEMILLA_SIMULACION = 2026

# Suma ponderada de medianas: estimación central, no media de la muestra.
retorno_central_pct = (
    (propuesta.asignacion_m * propuesta.roi_mediano_pct).sum()
    / CAPITAL_TOTAL_M)
resultado_central_m = CAPITAL_TOTAL_M * retorno_central_pct / 100
print('ROI central ponderado de medianas:',
      f'{retorno_central_pct:.1f}%')
print('Resultado ilustrativo sobre USD 500 M:',
      f'USD {resultado_central_m:.1f} M')

generador = np.random.default_rng(SEMILLA_SIMULACION)
retornos_independientes = []
retornos_compartidos = []

for _ in range(N_SIMULACIONES):
    resultado_independiente_m = 0
    resultado_compartido_m = 0
    for i in range(len(propuesta)):
        fila = propuesta.iloc[i]
        roi_del_grupo = retornos_historicos[fila['grupo']]
        muestras = generador.choice(
            roi_del_grupo, size=int(fila['proyectos']), replace=True)
        resultado_independiente_m += (
            fila['capital_por_proyecto_m'] * muestras.sum())
        resultado_compartido_m += (
            fila['asignacion_m'] * generador.choice(roi_del_grupo))
    retornos_independientes.append(
        resultado_independiente_m / CAPITAL_TOTAL_M)
    retornos_compartidos.append(
        resultado_compartido_m / CAPITAL_TOTAL_M)

retornos_independientes = pd.Series(retornos_independientes)
retornos_compartidos = pd.Series(retornos_compartidos)
escenarios = pd.DataFrame([
    {'supuesto':'Proyectos sorteados por separado',
     'prob_perder_capital_pct':100 * (retornos_independientes < 0).mean(),
     'roi_decil_inferior_pct':100 * retornos_independientes.quantile(0.10)},
    {'supuesto':'Resultado compartido dentro de cada grupo',
     'prob_perder_capital_pct':100 * (retornos_compartidos < 0).mean(),
     'roi_decil_inferior_pct':100 * retornos_compartidos.quantile(0.10)}
])
print('Resultados de', N_SIMULACIONES, 'simulaciones por escenario:')
display(escenarios.round(1))

**Decisión 5.2:** destinamos USD 170 M a **una** acción secuela grande (presupuesto mediano USD 163,8 M), USD 120 M a **una** animación secuela grande (USD 112,9 M), USD 100 M a **dos** acciones secuela medianas (USD 50 M por proyecto frente a una mediana de USD 47,9 M), USD 47 M a **una** comedia secuela mediana (mediana USD 43,8 M) y USD 63 M a **doce** películas de terror pequeñas (USD 5,25 M por proyecto frente a una mediana cercana a USD 5,2 M). Elegimos doce proyectos pequeños para repartir ese tramo entre varias películas; la cantidad supone que existan suficientes proyectos comparables durante los tres años de inversión.

La suma es **USD 500 M para 17 proyectos**. Acción y comedia de presupuesto medio reciben USD 147 M (29,4 %): las incluimos por la comparación de 5.1, pero acotamos su exposición porque sus ROI medianos siguen siendo negativos. Las dos producciones grandes reciben USD 290 M (58 %); aunque los doce proyectos de terror diversifican la cantidad de películas, **persiste una concentración importante del capital** en dos títulos. Los presupuestos medianos de los 17 proyectos suman USD 478,3 M. La diferencia de USD 21,7 M es un margen agregado pequeño y no está distribuida por igual: terror recibe apenas alrededor de 1,5 % más por proyecto que su presupuesto mediano. Antes de invertir habría que verificar presupuestos concretos, marketing y condiciones de cada contrato.

## 5.3 Retorno central y probabilidad de perder capital

Como cifra central usamos los **ROI medianos** de los cinco grupos, ponderados por los dólares asignados. Es un escenario basado en un caso típico de cada grupo, no la mediana exacta ni la esperanza matemática del portafolio. Para estimar el riesgo hacemos **10.000 simulaciones**, cantidad suficiente para obtener una frecuencia ilustrativa sin depender de una sola corrida; la semilla fija permite reproducir la tabla y no representa un año pronosticado. En cada proyecto sorteamos, con reposición, uno de los ROI históricos de su grupo. La pérdida se cuenta cuando el ROI total simulado es menor que cero, es decir, cuando el resultado modelado queda por debajo de los USD 500 M iniciales.

El primer escenario sortea cada proyecto por separado. El segundo hace que todos los proyectos de un mismo grupo compartan un único ROI sorteado: es una prueba de concentración, no una estimación medida de correlación real. En ambos casos usamos el ROI estimado de salas de 2.2 como aproximación; con estos datos no se puede calcular la probabilidad contractual de pérdida del fondo.

In [ ]:
no_invertir = []
for genero in ['Drama', 'Policial']:
    grupo = portafolio_base.loc[
        (portafolio_base.genero == genero)
        & (portafolio_base.tramo_presupuesto == '30-70')
        & (portafolio_base.es_secuela == 'No')]
    no_invertir.append({
        'grupo':genero + ', 30-70 M, no secuela',
        'peliculas_historicas':len(grupo),
        'roi_mediano_pct':100 * grupo.roi_salas_estimado.median(),
        'fracaso_pct':100 * grupo.fracaso_25.mean()
    })

print('Grupos que evitaríamos inicialmente:')
display(pd.DataFrame(no_invertir).round(1))

# Movemos los cortes para ver cuánto depende de ellos la tesis.
filas_sensibilidad_cortes = []
for corte_bajo, corte_medio, corte_alto in [(8,25,65), (10,30,70),
                                             (12,35,75)]:
    terror_bajo = portafolio_base.loc[
        (portafolio_base.genero == 'Terror')
        & (portafolio_base.es_secuela == 'No')
        & (portafolio_base.presupuesto_millones <= corte_bajo)]
    accion_media = portafolio_base.loc[
        (portafolio_base.genero == 'Acción')
        & (portafolio_base.es_secuela == 'Si')
        & (portafolio_base.presupuesto_millones > corte_medio)
        & (portafolio_base.presupuesto_millones <= corte_alto)]
    filas_sensibilidad_cortes.append({
        'cortes_millones':f'{corte_bajo}/{corte_medio}/{corte_alto}',
        'terror_n':len(terror_bajo),
        'terror_roi_mediano_pct':100 * terror_bajo.roi_salas_estimado.median(),
        'accion_n':len(accion_media),
        'accion_roi_mediano_pct':100 * accion_media.roi_salas_estimado.median()
    })

print('Sensibilidad de dos grupos a los cortes de presupuesto:')
display(pd.DataFrame(filas_sensibilidad_cortes).round(1))

**Lectura 5.3:** ponderar las cinco medianas por capital da un ROI central de **16,1 %**, equivalente a unos **USD 80,7 M** de resultado modelado sobre USD 500 M. En 10.000 simulaciones, el resultado queda debajo del capital inicial en **3,2 %** de los casos cuando se sortean los proyectos por separado y en **19,0 %** cuando los de un mismo grupo comparten resultado. El décimo percentil es **+19,4 %** en el primer escenario y **−15,8 %** en el segundo: en una de cada diez corridas aproximadamente el ROI queda por debajo de ese umbral.

Que el décimo percentil del primer escenario supere el 16,1 % no es un error: el 16,1 % combina las *medianas* de los grupos, mientras que la simulación vuelve a sortear todos los ROI históricos, incluidos éxitos extremos; además, son estadísticas de distribuciones distintas. La brecha de riesgo entre escenarios muestra cuánto influye el supuesto de independencia. Ninguna cifra garantiza el resultado futuro ni incorpora contratos, marketing real o ingresos fuera de salas.

## 5.4 Dónde no invertir y límites de la tesis

Evitamos inicialmente dos grupos de presupuesto medio sin secuela porque tienen muestras numerosas, ROI medianos muy negativos y alta frecuencia de fracaso comercial. No afirmamos que *todas* esas películas pierdan dinero. También dejamos fuera de la tesis principal las de presupuesto inferior a USD 1 M hasta verificar su escala caso por caso: en 3.2 encontramos tanto errores como éxitos genuinos en ese rango.

Para ver si la elección depende de los tramos, probamos cortes cercanos a los iniciales: 8/25/65 y 12/35/75 M frente a 10/30/70 M. Son cambios redondos de sensibilidad, no nuevos umbrales elegidos para mejorar el resultado. Mostramos dos grupos de la propuesta que permiten ver el efecto sobre la mediana y el tamaño de muestra.

**Recomendación condicionada 5.4:** priorizaría los cinco grupos propuestos para evaluar proyectos concretos y evitaría inicialmente **dramas y policiales no secuelas de USD 30–70 M**. Tienen 253 y 121 antecedentes, ROI medianos de **−61,6 %** y **−60,0 %**, y fracaso comercial de **68,4 %** y **75,2 %**, respectivamente. Tampoco asignaría capital a películas de menos de USD 1 M sin comprobar cada presupuesto, porque la base mezcla errores de escala con éxitos reales.

La recomendación es sensible a los cortes: al pasar de 8/25/65 a 12/35/75 M, la mediana de terror barato cambia de **80,6 % a 27,8 %**, mientras la de acción secuela de presupuesto medio pasa de **+2,4 % a −11,1 %**. Además, seleccionamos grupos mirando su desempeño histórico, lo que puede favorecer resultados difíciles de repetir. Faltan muchos presupuestos y los USD 500 M futuros no están en dólares de 2020. El modelo tampoco conoce contratos, participación real en los ingresos, marketing exacto, streaming, correlaciones reales ni la oferta de proyectos de los próximos tres años; conseguir doce títulos de terror comparables exige verificarla. Por eso el retorno central y las frecuencias de simulación sirven para discutir escenarios con el comité, **no para prometer rentabilidad o estimar el retorno contractual del fondo**.